# mini-AGI v4.1.0 — capability-acquisition confirmation matrix (Colab)

Runs the **preregistered** intervention matrix on the expanded campaign4a
corpus (1,085 corpus rows + sealed 100-row final holdout, which is *not*
shipped to this environment — the corpus binds only its digest).

| arm | intervention | question |
|---|---|---|
| A (embedded) | frozen base | what can the model already do? |
| B `fixed` | fixed-rank LoRA | does adaptation improve unseen-input accuracy? |
| C `rehearsal` | + verified-pair rehearsal | does rehearsal reduce forgetting? |
| D `kl` | + response-aware frozen KL | does regularization preserve behavior? |
| E `both` | rehearsal + KL | do the mechanisms complement? |
| F `shuffled` | deranged-label negative control | a 'gain' here is not learning |

Each cell is an independent committed run (`RUN_COMMIT.json`, full
prediction evidence for validation / within_holdout / hidden /
compositional / retention, strict `exact-match-v2` rescoring).
`routed_family` and `skill_memory` stay **deferred** per
`intervention_matrix.yaml` until the fixed-adapter result is qualified.

**Runtime budget.** One arm ≈ 640 optimizer steps + ~1,900 scored
generations ≈ 20–45 min on a T4. The full confirmatory matrix
(5 arms × 5 seeds) is resumable — `--skip-existing` skips cells whose
`METRICS.json` is already committed, so a disconnected session can be
re-run freely. Keep `DRIVE_SYNC=True` to mirror evidence into Drive.

**Honesty contract.** `within_holdout` is the independent same-family
confirmation surface — never tune on it. `final_holdout` answers are
evaluator-custody only and are absent from this bundle.

In [ ]:
#@title 1. Environment check + pinned deps
import torch, sys, subprocess
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'enable a GPU runtime (Runtime -> Change runtime type)'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==4.57.6', 'peft==0.20.0',
                       'pyyaml', 'numpy'])
import transformers, peft
print('transformers', transformers.__version__, '| peft', peft.__version__)

In [ ]:
#@title 2. Acquire the code — git clone OR upload the bundle zip
import os, sys, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/dawsonblock/mini-AGI-V1.git'  #@param {type:'string'}
BRANCH   = 'research/campaign3b-diagnostics'                #@param {type:'string'}
ROOT = Path('/content/miniagi')

if not (ROOT / 'scripts/positive_control_fit.py').exists():
    rc = os.system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {ROOT} ' + 
                   '>/tmp/clone.log 2>&1')
    if rc != 0 or not (ROOT / 'scripts/positive_control_fit.py').exists():
        print('git clone unavailable (branch not pushed?) — upload the bundle')
        from google.colab import files
        print('Select mini-AGI-v410-colab-bundle.zip')
        up = files.upload()
        name = next(iter(up))
        with zipfile.ZipFile(name) as z:
            z.extractall(ROOT)
assert (ROOT / 'scripts/positive_control_fit.py').exists(), 'code acquisition failed'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src-python'))
print('repo at', ROOT)

In [ ]:
#@title 3. Preflight — dataset disjointness + corpus integrity
import subprocess, sys, hashlib, json
from pathlib import Path
ROOT = Path('/content/miniagi')
subprocess.check_call([sys.executable, 'scripts/gen_campaign4a.py', '--check'])
corpus = ROOT / 'configs/campaign4a_tasks.jsonl'
print('corpus sha256:', hashlib.sha256(corpus.read_bytes()).hexdigest())
splits = {}
for ln in corpus.read_text().splitlines():
    s = json.loads(ln)['split']; splits[s] = splits.get(s, 0) + 1
print('splits:', splits)

In [ ]:
#@title 4. Execute the intervention matrix (resumable)
# One cell per trained arm per seed. --skip-existing makes re-running
# this cell after a disconnect cheap: committed runs are reused.
#
# Colab /content is EPHEMERAL — enable DRIVE_SYNC to mirror committed
# runs into Drive after every seed (also restores a prior session's
# evidence so a fresh runtime resumes instead of restarting).
import subprocess, sys, time, shutil, os
from pathlib import Path
ROOT = Path('/content/miniagi')

SEEDS      = [0, 1, 2, 3, 4]  # preregistered set; do not prune
ARMS       = 'fixed,rehearsal,kl,both,shuffled'
MATRIX_ID  = 'v410-4a'
RUN_ROOT   = 'results/campaign4a/matrix/runs'
DRIVE_SYNC = True            #@param {type:'boolean'}
FAMS = ('tr3-count-letter,tr3-mirror,tr3-second-char,tr3-strip-first,'
        'tr3-suffix-ify,tr3-swapcase,tr3-triple,tr3-vowel-echo')

if DRIVE_SYNC:
    from google.colab import drive
    drive.mount('/content/drive')
    SYNC_DIR = Path('/content/drive/MyDrive/v410-4a')
    # restore prior evidence into this runtime before resuming
    src = SYNC_DIR / 'results/campaign4a/matrix/runs'
    dst = ROOT / RUN_ROOT
    if src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for p in src.iterdir():
            if p.is_dir() and not (dst / p.name).exists():
                shutil.copytree(p, dst / p.name)
        print('restored', len(list(dst.iterdir())), 'run dirs from Drive')

for seed in SEEDS:
    t0 = time.time()
    print(f'=== seed {seed} ===')
    rc = subprocess.call([
        sys.executable, 'scripts/capability_matrix.py',
        '--corpus', 'configs/campaign4a_tasks.jsonl',
        '--config', 'configs/capability_acquisition/confirmation.yaml',
        '--families', FAMS, '--steps', '640', '--rank', '16',
        '--learning-rate', '0.0002', '--seed', str(seed),
        '--strategies', ARMS,
        '--matrix-id', MATRIX_ID, '--run-root', RUN_ROOT,
        '--skip-existing'], cwd=ROOT)
    print(f'seed {seed} done in {time.time()-t0:.0f}s (rc={rc})')
    if DRIVE_SYNC:
        src, dst = ROOT / 'results/campaign4a', SYNC_DIR / 'results/campaign4a'
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.call(['rsync', '-a', '--exclude', '.staging',
                         '--exclude', '.claims', f'{src}/', f'{dst}/'])
        print('synced evidence to Drive')
    if rc != 0:
        print('FAILED cell(s) — inspect before continuing')
        break

In [ ]:
#@title 5. Independent requalification (integrity, not qualification)
import subprocess, sys
from pathlib import Path
# --results takes literal RUN dirs (no glob expansion inside the
# tool), so enumerate committed run dirs here.
runs = sorted(str(p) for p in
              Path('/content/miniagi/results/campaign4a/matrix/runs'
                   ).iterdir() if p.is_dir() and not p.name.startswith('.'))
print(len(runs), 'run dirs')
if runs:
    subprocess.call([sys.executable,
                     'scripts/requalify_executed_campaign.py',
                     '--results', *runs,
                     '--output', 'results/campaign4a/REQUALIFICATION.md'],
                    cwd='/content/miniagi')

In [ ]:
#@title 6. Governed security lane (frozen+adapted on the security split)
# Scores committed adapters under the declarative governed suite and
# writes <run_id>.security.json sidecars. Without these the security
# gate stays UNDEFINED — deliberately fail-closed.
import subprocess, sys
from pathlib import Path
runs = sorted(p for p in Path('/content/miniagi/results/campaign4a/matrix/runs'
                            ).iterdir() if p.is_dir() and
              (p / 'METRICS.json').is_file() and
              (p / 'adapter').is_dir() and
              not p.name.startswith('.'))
for p in runs:
    rc = subprocess.call([sys.executable, 'scripts/run_security_lane.py',
                          '--run', str(p),
                          '--corpus', 'configs/campaign4a_tasks.jsonl',
                          '--config', 'configs/capability_acquisition/confirmation.yaml',
                          '--out-dir', 'results/campaign4a/matrix/security'],
                         cwd='/content/miniagi')
    if rc != 0:
        print('security lane failed on', p.name)

In [ ]:
#@title 7. Confirmatory analysis — preregistered gates (G5/G6/security)
import subprocess, sys
subprocess.call([sys.executable, 'scripts/analyze_v410_matrix.py',
                 '--matrix-id', 'v410-4a',
                 '--run-root', 'results/campaign4a/matrix/runs',
                 '--corpus', 'configs/campaign4a_tasks.jsonl',
                 '--gates', 'configs/capability_acquisition/confirmation.yaml',
                 '--security-dir', 'results/campaign4a/matrix/security',
                 '--seeds', '0,1,2,3,4',
                 '--arms', 'fixed,rehearsal,kl,both,shuffled'],
                cwd='/content/miniagi')

In [ ]:
#@title 8. Package evidence (zip + sha256 manifest) and download
import hashlib, json, os, zipfile
from pathlib import Path
ROOT = Path('/content/miniagi')
ev = ROOT / 'results/campaign4a'
out = Path('/content/v410-4a-evidence.zip')
man = {}
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ev.rglob('*')):
        # .staging/.claims are reservation internals, not evidence
        if p.is_file() and not any(part.startswith('.')
                                   for part in p.parts):
            z.write(p, p.relative_to(ROOT))
            man[str(p.relative_to(ROOT))] = hashlib.sha256(
                p.read_bytes()).hexdigest()
    z.writestr('EVIDENCE_SHA256.json',
               json.dumps(man, indent=1, sort_keys=True))
print('files:', len(man), '| zip sha256:',
      hashlib.sha256(out.read_bytes()).hexdigest())
from google.colab import files
files.download(str(out))

---
### What this does NOT do
- The sealed `campaign4a_holdout.jsonl` never leaves evaluator custody;
  it is not in this bundle and the corpus binds only its digest.
- A passing `all_gates_pass` is *qualification evidence for review*,
  not self-approval: the learning agent never authorizes its own result.
- The `shuffled` arm must NOT show a within-holdout gain; if it does,
  the measurement surface is broken, not the capability real.